<a href="https://colab.research.google.com/github/leonrichandra/bk/blob/main/facilitator_complete_colab_workshop_boyolali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop Data Science: Mengolah Data Menjadi Cerita Bisnis

**Organisasi:** Balik Kampoeng  
**Sekolah:** SMAN 1 Boyolali  
**Tanggal:** 29 Mei 2026  

## Misi workshop
Hari ini kita berperan sebagai **Junior Data Scientist** untuk membantu sebuah usaha susu lokal di Boyolali.
Kita akan mengubah data mentah menjadi laporan bisnis yang rapi, informatif, dan bisa dipresentasikan.

> Catatan: Dataset ini adalah **synthetic dataset** untuk latihan, tetapi dibuat agar dekat dengan konteks Boyolali dan bisnis produk susu.

## 0. Setup library
Kita menggunakan library Python yang umum dipakai dalam data science:

- `pandas` untuk membaca dan mengolah data
- `matplotlib` dan `seaborn` untuk visualisasi
- `plotly` untuk grafik interaktif
- `sklearn` untuk model prediksi sederhana

In [ ]:
# Import library yang akan kita pakai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Supaya angka Rupiah lebih mudah dibaca
def format_rupiah(x):
    return f"Rp{x:,.0f}".replace(',', '.')

pd.set_option('display.max_columns', 50)

## 1. Load dataset

Jika dataset sudah di-upload ke GitHub, ganti nilai `DATA_URL` dengan **Raw GitHub URL**.
Jika belum, upload file `dataset_penjualan_susu_boyolali.csv` ke Colab lalu jalankan cell ini.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/leonrichandra/bk/refs/heads/main/dataset_penjualan_susu_boyolali.csv"
FILE_NAME = "dataset_penjualan_susu_boyolali.csv"

if DATA_URL:
    df_raw = pd.read_csv(DATA_URL)
else:
    df_raw = pd.read_csv(FILE_NAME)

print("Dataset berhasil dibaca!")
df_raw.head()

## 2. Eksplorasi awal: mengenal data mentah

Sebelum analisis, kita harus paham dulu struktur data.
Pertanyaan awal:

1. Ada berapa baris dan kolom?
2. Kolom apa saja yang tersedia?
3. Apakah ada data kosong?
4. Apakah ada data duplikat?

In [ ]:
# Melihat ukuran dataset: (jumlah_baris, jumlah_kolom)
df_raw.shape

In [ ]:
# Melihat daftar kolom dan tipe data
# object biasanya berarti teks, int/float berarti angka
df_raw.info()

In [ ]:
# Melihat statistik sederhana untuk kolom angka
df_raw.describe()

In [ ]:
# Mengecek jumlah data kosong di setiap kolom
df_raw.isnull().sum()

In [ ]:
# Mengecek jumlah baris duplikat
df_raw.duplicated().sum()

## 3. Data cleaning sederhana

Data dunia nyata jarang langsung rapi. Di bagian ini kita akan memperbaiki masalah umum:

- format tanggal
- nama produk yang tidak konsisten
- harga satuan yang kosong
- baris duplikat

In [ ]:
# Buat copy agar data mentah tetap aman
df = df_raw.copy()

# 1) Ubah kolom tanggal menjadi format tanggal Python
df['tanggal'] = pd.to_datetime(df['tanggal'])

# 2) Rapikan nama produk: hilangkan spasi, buat format judul
df['produk'] = df['produk'].astype(str).str.strip().str.title()

# Perbaiki beberapa nama produk yang butuh kapitalisasi khusus
df['produk'] = df['produk'].replace({
    'Susu Uht': 'Susu UHT',
    'Susu Segar': 'Susu Segar',
    'Keju Mozzarella': 'Keju Mozzarella',
    'Permen Susu': 'Permen Susu',
    'Es Krim': 'Es Krim'
})

# 3) Isi harga_satuan kosong dengan median harga berdasarkan produk
median_harga_per_produk = df.groupby('produk')['harga_satuan'].transform('median')
df['harga_satuan'] = df['harga_satuan'].fillna(median_harga_per_produk)

# 4) Hitung ulang total_penjualan setelah harga diperbaiki
df['total_penjualan'] = df['jumlah_terjual'] * df['harga_satuan']

# 5) Hapus baris duplikat
df = df.drop_duplicates()

print("Data setelah cleaning:")
print(df.shape)
df.head()

In [ ]:
# Cek ulang: seharusnya data kosong dan duplikat sudah berkurang/hilang
print("Missing values:")
print(df.isnull().sum())
print("\nJumlah duplikat:", df.duplicated().sum())

## 4. Feature engineering: membuat kolom baru

Feature engineering artinya membuat kolom baru yang membantu analisis.
Di sini kita buat kolom bulan, hari dalam minggu, dan flag angka untuk promo/libur.

In [ ]:
df['bulan'] = df['tanggal'].dt.to_period('M').astype(str)
df['nama_hari'] = df['tanggal'].dt.day_name()
df['akhir_pekan'] = df['tanggal'].dt.weekday >= 5

# Buat flag angka agar bisa dipakai untuk correlation dan model prediksi
df['promo_flag'] = (df['promo_aktif'] == 'Ya').astype(int)
df['hari_libur_flag'] = (df['hari_libur'] == 'Ya').astype(int)

df[['tanggal', 'bulan', 'nama_hari', 'akhir_pekan', 'promo_flag', 'hari_libur_flag']].head()

## 5. Analisis bisnis: menjawab pertanyaan Pak Bambang

Sekarang kita mulai menjawab pertanyaan bisnis:

1. Produk apa yang menghasilkan penjualan terbesar?
2. Kecamatan mana yang paling kuat?
3. Apakah promo benar-benar membantu?
4. Apakah ada pola khusus saat hari libur?

In [ ]:
# Total penjualan dan jumlah terjual berdasarkan produk
analisis_produk = (
    df.groupby('produk')
      .agg(total_qty=('jumlah_terjual', 'sum'),
           total_penjualan=('total_penjualan', 'sum'),
           rata_rata_harga=('harga_satuan', 'mean'))
      .sort_values('total_penjualan', ascending=False)
)

analisis_produk['total_penjualan_rp'] = analisis_produk['total_penjualan'].apply(format_rupiah)
analisis_produk

In [ ]:
# Analisis kecamatan: siapa pasar terbesar?
analisis_kecamatan = (
    df.groupby('kecamatan')
      .agg(total_qty=('jumlah_terjual', 'sum'),
           total_penjualan=('total_penjualan', 'sum'),
           rata_rating=('rating_pelanggan', 'mean'))
      .sort_values('total_penjualan', ascending=False)
)
analisis_kecamatan['total_penjualan_rp'] = analisis_kecamatan['total_penjualan'].apply(format_rupiah)
analisis_kecamatan

In [ ]:
# Multi-level groupby: penjualan per kecamatan dan produk
pivot_kecamatan_produk = pd.pivot_table(
    df,
    index='kecamatan',
    columns='produk',
    values='jumlah_terjual',
    aggfunc='sum',
    fill_value=0
)

pivot_kecamatan_produk

In [ ]:
# Dampak promo berdasarkan jenis pelanggan
promo_effect = (
    df.groupby(['jenis_pelanggan', 'promo_aktif'])
      .agg(rata_qty=('jumlah_terjual', 'mean'),
           total_penjualan=('total_penjualan', 'sum'),
           jumlah_transaksi=('jumlah_terjual', 'count'))
      .reset_index()
)

promo_effect

## 6. Visualisasi: membuat data lebih mudah dipahami

Visualisasi membantu kita melihat pola dengan cepat.

In [ ]:
# Visualisasi 1: total penjualan per produk
plt.figure(figsize=(9,5))
analisis_produk['total_penjualan'].sort_values().plot(kind='barh')
plt.title('Total Penjualan berdasarkan Produk')
plt.xlabel('Total Penjualan (Rupiah)')
plt.ylabel('Produk')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 2: tren penjualan bulanan
tren_bulanan = df.groupby('bulan')['total_penjualan'].sum().reset_index()

plt.figure(figsize=(11,5))
plt.plot(tren_bulanan['bulan'], tren_bulanan['total_penjualan'], marker='o')
plt.title('Tren Total Penjualan per Bulan')
plt.xlabel('Bulan')
plt.ylabel('Total Penjualan (Rupiah)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 3: grafik interaktif dengan Plotly
fig = px.bar(
    df.groupby(['kecamatan', 'produk'], as_index=False)['jumlah_terjual'].sum(),
    x='kecamatan',
    y='jumlah_terjual',
    color='produk',
    title='Jumlah Terjual per Kecamatan dan Produk',
    barmode='group'
)
fig.show()

## 7. Correlation heatmap: mencari variabel yang bergerak bersama

Correlation membantu melihat hubungan antar angka.
Nilai mendekati `1` berarti naik bersama, mendekati `-1` berarti bergerak berlawanan.

In [ ]:
kolom_numerik = [
    'jumlah_terjual', 'harga_satuan', 'total_penjualan',
    'suhu_udara_celsius', 'promo_flag', 'hari_libur_flag',
    'stok_awal', 'stok_sisa', 'jarak_pengiriman_km',
    'biaya_distribusi', 'rating_pelanggan'
]

corr = df[kolom_numerik].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Fokus ke Susu Segar: apakah suhu udara berhubungan dengan jumlah terjual?
susu_segar = df[df['produk'] == 'Susu Segar']

plt.figure(figsize=(8,5))
plt.scatter(susu_segar['suhu_udara_celsius'], susu_segar['jumlah_terjual'], alpha=0.5)
plt.title('Suhu Udara vs Jumlah Terjual - Susu Segar')
plt.xlabel('Suhu Udara (°C)')
plt.ylabel('Jumlah Terjual')
plt.tight_layout()
plt.show()

## 8. Model prediksi sederhana: Linear Regression

Di bagian ini kita membuat model sederhana untuk memprediksi jumlah penjualan **Susu Segar**.
Kita tidak mengejar model sempurna. Tujuannya adalah memahami ide dasar:

> komputer belajar pola dari data lama, lalu membuat prediksi untuk data baru.

In [ ]:
# Ambil data Susu Segar saja
model_df = df[df['produk'] == 'Susu Segar'].copy()

# Fitur/input yang dipakai untuk prediksi
X = model_df[['suhu_udara_celsius', 'harga_satuan', 'promo_flag', 'hari_libur_flag']]

# Target/output yang ingin diprediksi
y = model_df['jumlah_terjual']

# Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Buat dan latih model
model = LinearRegression()
model.fit(X_train, y_train)

# Prediksi pada data uji
prediksi = model.predict(X_test)

print('MAE:', round(mean_absolute_error(y_test, prediksi), 2))
print('R2 Score:', round(r2_score(y_test, prediksi), 2))

In [ ]:
# Contoh prediksi: hari panas, tidak promo, bukan hari libur
contoh_baru = pd.DataFrame({
    'suhu_udara_celsius': [31.0],
    'harga_satuan': [12000],
    'promo_flag': [0],
    'hari_libur_flag': [0]
})

hasil_prediksi = model.predict(contoh_baru)[0]
print(f"Prediksi penjualan Susu Segar: {hasil_prediksi:.0f} unit")

## 9. Anomaly detection: menemukan data yang aneh

Anomaly detection membantu kita menemukan data yang berbeda jauh dari biasanya.
Contoh: salah input, permintaan tiba-tiba melonjak, atau kejadian khusus.

In [ ]:
# Hitung z-score untuk total_penjualan
# Z-score tinggi berarti data jauh dari rata-rata
rata = df['total_penjualan'].mean()
std = df['total_penjualan'].std()
df['z_score_penjualan'] = (df['total_penjualan'] - rata) / std

anomali = df[df['z_score_penjualan'].abs() > 3].sort_values('z_score_penjualan', ascending=False)

anomali[['tanggal', 'kecamatan', 'produk', 'jumlah_terjual', 'harga_satuan', 'total_penjualan', 'z_score_penjualan']].head(10)

## 10. Kesimpulan laporan

Contoh cara menulis insight yang baik:

1. **Temuan:** Produk dengan penjualan terbesar adalah ...
2. **Bukti data:** Berdasarkan total penjualan, produk ini menghasilkan ...
3. **Makna bisnis:** Produk ini perlu dijaga stoknya, terutama pada ...
4. **Rekomendasi:** Pak Bambang sebaiknya ...

In [ ]:
# Membuat ringkasan otomatis sederhana dari hasil analisis
produk_teratas = analisis_produk.index[0]
kecamatan_teratas = analisis_kecamatan.index[0]
bulan_teratas = tren_bulanan.sort_values('total_penjualan', ascending=False).iloc[0]['bulan']

print('RINGKASAN ANALISIS')
print('1. Produk dengan total penjualan tertinggi:', produk_teratas)
print('2. Kecamatan dengan total penjualan tertinggi:', kecamatan_teratas)
print('3. Bulan dengan total penjualan tertinggi:', bulan_teratas)
print('4. Jumlah data anomali yang perlu dicek:', len(anomali))

## Rekomendasi untuk Pak Bambang

Silakan lengkapi bagian ini berdasarkan hasil analisis kalian:

1. Produk prioritas untuk dijaga stoknya adalah **...** karena **...**
2. Kecamatan yang paling potensial untuk ekspansi adalah **...** karena **...**
3. Promo paling efektif untuk jenis pelanggan **...**, sehingga strategi promo sebaiknya **...**
4. Data anomali pada tanggal **...** perlu dicek ulang karena **...**

## Cara export menjadi PDF
Di Google Colab: **File → Print → Save as PDF**.